In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import(
    col,
    row_number,
    trim
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

#### Importing validation class from the test_utils module

In [0]:
from validation_utils.test_utils import Validations
obj = Validations()

#### Defining orchestrator function that runs all the test functions inside the test_utils

In [0]:
def orchestrate_tests(
    table: str,
    obj: Validations,
    primary_col: str,
    dedup_cols: list,
    cdc: str
) -> None:
    df = spark.table(f"lakehouse.silver.{table}")

    def log(msg: str) -> None:
        print(f"\n{'-' * 20}\t{msg}\t{'-' * 20}\n")

    log("Null Check Start")
    obj.null_check(df, primary_col)
    log("Null Check End")

    log("Duplicate Check Start")
    obj.duplicate_check(df, dedup_cols, cdc)
    log("Duplicate Check End")

    log("Whitespace Check Start")
    obj.whitespace_check(df)
    log("Whitespace Check End")



##### 1. crm_customers

In [0]:
orchestrate_tests(
    table = "crm_customers",
    obj = obj,
    primary_col = 'customer_id',
    dedup_cols = ['customer_id'],
    cdc = 'create_date'
)

#### 2. crm_products

In [0]:
orchestrate_tests(
    table = "crm_products",
    obj = obj,
    primary_col = 'product_id',
    dedup_cols = ['product_id'],
    cdc = 'start_date'
)

In [0]:
table_name_product = "crm_products"

##### Checking if there are nulls in product_cost

In [0]:
if spark.sql(f"""
            SELECT *
            FROM lakehouse.silver.{table_name_product}
            WHERE product_cost IS NULL
        """).count() > 0:
    raise ValueError(f"Null value found in {table_name_product}.product_cost")
else:
    print(f"No nulls found in {table_name_product}.product_cost")

##### Checking if end_date < start_date

In [0]:
if spark.sql(f"""
            SELECT *
            FROM lakehouse.silver.{table_name_product}
            WHERE end_date < start_date
               """).count() > 0:
    raise Exception(f"End date is less than start date in {table_name_product}")
else:
    print(f"End date and start date are valid in {table_name_product}")

#### 3. crm_sales

In [0]:
orchestrate_tests(
    table = "crm_sales",
    obj = obj,
    primary_col = "order_number",
    dedup_cols = ["order_number", 'product_number', 'customer_id'],
    cdc = 'order_date'
)

##### Checking the missing records in other tables based on the foreign keys of crm_sales
- Since this is a transactional table used for making fact table. So, check whether it can be used for joins or not by checking the missing records
- If no missing records then useful for join

In [0]:
table_name_sales = "crm_sales"

In [0]:
if spark.sql(f"""
            SELECT *
            from lakehouse.silver.{table_name_sales}
            WHERE customer_id NOT IN (
                    SELECT customer_id
                    FROM lakehouse.silver.crm_customers
                )
            """).count() > 0:
    raise Exception(f"Invalid customer_id found in {table_name_sales}")
else:
    print(f"No invalid customer_id found in {table_name_sales}")

In [0]:
if spark.sql(f"""
            SELECT *
            FROM lakehouse.silver.{table_name_sales}
            WHERE product_number NOT IN (
                    SELECT product_number
                    FROM lakehouse.silver.crm_products
                )
             """).count() > 0:
    raise Exception(f"Invalid product_number found in {table_name_sales}")
else:
    print(f"No invalid product_number found in {table_name_sales}")

##### Checking dates

In [0]:
if spark.sql(f"""
            SELECT *
            FROM lakehouse.silver.{table_name_sales}
            WHERE order_date > due_date OR order_date > ship_date
             """).count() > 0:
    raise ValueError(f"Invalid order_date found in {table_name_sales}")
else:
    print(f"No invalid order_date found in {table_name_sales}")

##### Checking sales_amount, quantity and price

- Checking sales_amount

In [0]:
if spark.sql("""
            SELECT *
            FROM lakehouse.silver.crm_sales
            WHERE sales_amount IS NULL OR sales_amount < 0 OR sales_amount != price * quantity
             """).count() > 0:
    raise ValueError(f"Invalid sales_amount found in {table_name_sales}")
else:
    print(f"Sales amount is valid in {table_name_sales}") 

- Checking price

In [0]:
if spark.sql("""
            SELECT *
            FROM lakehouse.silver.crm_sales
            WHERE price IS NULL OR price < 0 OR price != sales_amount / quantity
             """).count() > 0:
    raise ValueError(f"Invalid price found in {table_name_sales}")
else:
    print(f"Price is valid in {table_name_sales}") 